In [1]:
import gymnasium as gym

env = gym.make("CartPole-v1", render_mode="rgb_array", max_episode_steps=1000)
obs, info = env.reset(seed=42)
obs

array([ 0.0273956 , -0.00611216,  0.03585979,  0.0197368 ], dtype=float32)

In [2]:
img = env.render()
img.shape

e:\Program\handson-mlp\.venv\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


(400, 600, 3)

In [3]:
env.action_space

Discrete(2)

In [5]:
action = 1
obs, reward, done, truncated, info = env.step(action)
obs

array([ 0.03104291,  0.38306385,  0.03102613, -0.5424507 ], dtype=float32)

In [ ]:
reward, done, truncated, info #h pos, v pos, angle, angular velocity

(1.0, False, False, {})

In [8]:
def basic_policy(obs):
    angle = obs[2]
    return 0 if angle < 0 else 1

totals = []
for episode in range(500):
    total_rewards = 0
    obs, info = env.reset(seed=episode)
    while True:
        action = basic_policy(obs)
        obs, reward, done, truncated, info = env.step(action)
        total_rewards += float(reward)
        if done or truncated: break
        
    totals.append(total_rewards)

In [9]:
import numpy as np
np.mean(totals), np.std(totals), min(totals), max(totals)

(np.float64(41.698), np.float64(8.389445512070509), 24.0, 63.0)

In [ ]:
import torch
import torch.nn as nn

class PolicyNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(4,5), nn.ReLU(), nn.Linear(5,1))
    
    def forward(self, state):
        return self.net(state)

def choose_action(model:PolicyNetwork, obs):
    state = torch.as_tensor(obs)
    logit = model(state)
    dist = torch.distributions.Bernoulli(logits=logit)
    action = dist.sample()
    log_prob = dist.log_prob(action)
    return int(action.item()), log_prob